### Paso 1: Carga de bibliotecas especializadas


In [19]:
import pandas as pd #manipulación de datos
import seaborn as sns #visualizaciones
from sklearn.model_selection import train_test_split #módulos de sklearn para construir y evaluar nuestro modelo predictivo.
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

### Paso 2: Importación remota del conjunto de datos


In [20]:
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)

In [21]:
df.shape

(7043, 21)

In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [23]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### Paso 3: Limpieza de datos y transformación de variables
Los modelos matemáticos necesitan números para funcionar. Aquí realizamos dos tareas vitales:
1. Convertimos la columna `TotalCharges` a formato numérico (rellenando los valores faltantes con 0).
2. Transformamos nuestra variable objetivo `Churn`, cambiando el texto ('Yes' o 'No') por valores binarios (1 y 0).

In [10]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

### Paso 4: División de variables y preparación de datos
En esta etapa preparamos los datos para el entrenamiento:
- **X:** Variables independientes (eliminamos el `customerID` porque no predice nada, y el `Churn` porque es lo que queremos adivinar).
- Convertimos las demás variables de texto en columnas binarias (Variables Dummy) usando `pd.get_dummies`.
- **y:** Nuestra variable dependiente (Churn).
- Finalmente, dividimos los datos: 80% para que el modelo entrene y 20% para ponerlo a prueba.

In [24]:
X = pd.get_dummies(df.drop(columns=['customerID', 'Churn']))
y = df['Churn']

In [26]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [27]:
X_train

,SeniorCitizen,tenure,MonthlyCharges,gender_Female,gender_Male,Partner_No,Partner_Yes,Dependents_No,Dependents_Yes,PhoneService_No,...,TotalCharges_995.35,TotalCharges_996.45,TotalCharges_996.85,TotalCharges_996.95,TotalCharges_997.65,TotalCharges_997.75,TotalCharges_998.1,TotalCharges_999.45,TotalCharges_999.8,TotalCharges_999.9
2142,0,21,64.85,True,False,True,False,False,True,False,...,False,False,False,False,False,False,False,False,False,False
1623,0,54,97.20,True,False,True,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
6074,0,1,23.45,False,True,False,True,True,False,True,...,False,False,False,False,False,False,False,False,False,False
1362,0,4,70.20,False,True,True,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
6754,0,0,61.90,False,True,True,False,False,True,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3772,0,1,95.00,False,True,False,True,True,False,False,...,False,False,False,False,False,False,False,False,False,False
5191,0,23,91.10,True,False,False,True,False,True,False,...,False,False,False,False,False,False,False,False,False,False
5226,0,12,21.15,False,True,False,True,False,True,False,...,False,False,False,False,False,False,False,False,False,False
5390,1,12,99.45,False,True,True,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False


### Paso 5: Entrenamiento del modelo predictivo
Instanciamos el algoritmo de **Random Forest** (Bosque Aleatorio) configurándolo con 100 árboles de decisión. Luego, lo entrenamos (ajustamos) utilizando exclusivamente nuestros datos de entrenamiento (`X_train` y `y_train`).

In [28]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

### Paso 6: Generación de predicciones y métricas de evaluación
El momento de la verdad. Le pedimos al modelo entrenado que prediga si los clientes del conjunto de prueba abandonarán o no el servicio. Finalmente, imprimimos un reporte de métricas para evaluar su precisión y rendimiento.

In [29]:
predictions = model.predict(X_test)

In [30]:
print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

          No       0.82      0.91      0.86      1036
         Yes       0.64      0.46      0.53       373

    accuracy                           0.79      1409
   macro avg       0.73      0.68      0.70      1409
weighted avg       0.78      0.79      0.78      1409

